# Rearc Data Quest - Data Ingestion Pipeline

This notebook handles the ingestion of BLS and population data into Databricks volumes.

In [0]:
# Configuration widgets
# dbutils.widgets.text("catalog_name", "******")
# dbutils.widgets.text("schema_name", "******")
# dbutils.widgets.text("volume_name", "*******")

In [0]:
import sys
import os
dbutils.widgets.text("bundle_source_path", os.path.abspath(os.path.join(os.getcwd(),  '..', '..', 'src')))
project_source_path = dbutils.widgets.get("bundle_source_path")
sys.path.append(project_source_path)

In [0]:
# Get config values
catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
volume_name = dbutils.widgets.get("volume_name")

# Construct paths
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"
bls_path = f"{volume_path}/bls"
population_path = f"{volume_path}/population"
bls_tracker_path = f"{volume_path}/_tracking/bls_tracker.json"
population_tracker_path = f"{volume_path}/_tracking/population_tracker.json"

print(f"Volume Path: {volume_path}")
print(f"BLS Path: {bls_path}")
print(f"Population Path: {population_path}")

## Step 1: Ingest BLS Data

Download all files from the BLS productivity time-series directory.
Uses hash-based tracking to only download new or changed files.

In [0]:
from rearc_data_quest.ingestion import BLSDataIngestion

# Configuration
BLS_BASE_URL = "https://download.bls.gov/pub/time.series/pr/"
USER_AGENT = "dataquest@example.com"

# Initialize ingestion
bls_ingestion = BLSDataIngestion(BLS_BASE_URL, USER_AGENT)

# Ingest data with separate tracker
bls_stats = bls_ingestion.ingest_all(bls_path, bls_tracker_path)

print(f"\nBLS Ingestion Statistics:")
print(f"  Total files: {bls_stats['total_files']}")
print(f"  Downloaded: {bls_stats['downloaded']}")
print(f"  Skipped (unchanged): {bls_stats['skipped']}")
print(f"  Removed from source: {bls_stats['removed']}")
if bls_stats.get('timestamp_dir'):
    print(f"  Timestamp directory: {bls_stats['timestamp_dir']}")

## Step 2: Ingest Population Data

Fetch population data from the DataUSA API.

In [0]:
from rearc_data_quest.ingestion import PopulationDataIngestion

# Configuration
POPULATION_API_URL = (
    "https://honolulu-api.datausa.io/tesseract/data.jsonrecords"
    "?cube=acs_yg_total_population_1&drilldowns=Year%2CNation"
    "&locale=en&measures=Population"
)

# Initialize ingestion
pop_ingestion = PopulationDataIngestion(POPULATION_API_URL, USER_AGENT)

# Ingest data with separate tracker (note: now takes directory not full path)
pop_file, was_downloaded, timestamp = pop_ingestion.save_to_file(
    population_path,
    population_tracker_path
)

print(f"\nPopulation Data:")
if was_downloaded:
    print(f"  File: {pop_file}")
    print(f"  Downloaded: True")
    print(f"  Timestamp directory: {timestamp}")
else:
    print(f"  Downloaded: False (unchanged)")

## Result

The data has been ingested into the volume with timestamp-based directories for version control.

**Volume Structure:**
- `/bls/YYYYMMDD_HHMMSS/` - BLS files for each ingestion run
- `/population/YYYYMMDD_HHMMSS/` - Population files for each ingestion run
- `/_tracking/bls_tracker.json` - Tracks BLS file hashes for idempotency
- `/_tracking/population_tracker.json` - Tracks population file hash for idempotency

**Idempotency:**
- Re-running ingestion only downloads files with changed content
- Unchanged files are skipped
- Each download creates a new timestamp directory when there is file to download